# Taller — Punto 1: Modelado y visualización del brazo robótico 





In [83]:
%pip install numpy matplotlib roboticstoolbox-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\USUARIO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [84]:
import numpy as np
import matplotlib.pyplot as plt
from roboticstoolbox import DHRobot, RevoluteMDH
from spatialmath import SE3

%matplotlib tk
# Abre el gráfico en una ventana aparte: se puede arrastrar con el mouse para rotar.

## Construcción del robot con roboticstoolbox-python



In [85]:
L1, L2, L3, L4, L5 = 0.10, 0.15, 0.15, 0.04, 0.05  # metros

robot = DHRobot([
    RevoluteMDH(d=L1, a=0,  alpha=0),        # J1: rotación de base
    RevoluteMDH(d=0,  a=0,  alpha=np.pi/2),  # J2: hombro (shoulder pitch)
    RevoluteMDH(d=0,  a=L2, alpha=0),        # J3: codo (elbow pitch)
    RevoluteMDH(d=0,  a=L3, alpha=0),        # J4: muñeca - pitch (wrist pitch)
    RevoluteMDH(d=0,  a=L4, alpha=np.pi/2),  # J5: muñeca - roll (gira el gripper)
], name="Brazo 6 GDL", tool=SE3.Tz(L5))

print(robot)  # "RRRRR" = 5 articulaciones de revoluta (+ el gripper, GDL 6, aparte)

DHRobot: Brazo 6 GDL, 5 joints (RRRRR), dynamics, modified DH parameters
┌──────┬───────┬─────┬─────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │ θⱼ  │ dⱼ  │
├──────┼───────┼─────┼─────┤
│    0 │  0.0° │  q1 │ 0.1 │
│    0 │ 90.0° │  q2 │   0 │
│ 0.15 │  0.0° │  q3 │   0 │
│ 0.15 │  0.0° │  q4 │   0 │
│ 0.04 │ 90.0° │  q5 │   0 │
└──────┴───────┴─────┴─────┘

┌──────┬──────────────────────────────────────┐
│ tool │ t = 0, 0, 0.05; rpy/xyz = 0°, 0°, 0° │
└──────┴──────────────────────────────────────┘



## Cinemática directa


In [86]:
q0 = [0, 0, 0, 0, 0]  # brazo totalmente extendido (pose de reposo)

T0 = robot.fkine(q0)
print(T0)

   1         0         0         0.34      
   0        -1         0         0         
   0         0        -1         0.05      
   0         0         0         1         



## Graficado 3D 

In [87]:
def cilindro(ax, p0, p1, radio, color, n=14):
    p0, p1 = np.array(p0), np.array(p1)
    v = p1 - p0
    mag = np.linalg.norm(v)
    if mag < 1e-9:
        return
    v = v / mag
    no_v = np.array([1, 0, 0]) if abs(v[0]) < 0.9 else np.array([0, 1, 0])
    n1 = np.cross(v, no_v); n1 /= np.linalg.norm(n1)
    n2 = np.cross(v, n1)
    t, theta = np.meshgrid(np.linspace(0, mag, 2), np.linspace(0, 2 * np.pi, n))
    X = p0[0] + v[0]*t + radio*np.sin(theta)*n1[0] + radio*np.cos(theta)*n2[0]
    Y = p0[1] + v[1]*t + radio*np.sin(theta)*n1[1] + radio*np.cos(theta)*n2[1]
    Z = p0[2] + v[2]*t + radio*np.sin(theta)*n1[2] + radio*np.cos(theta)*n2[2]
    ax.plot_surface(X, Y, Z, color=color, shade=True, linewidth=0)


def esfera(ax, centro, radio, color):
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 14), np.linspace(0, np.pi, 10))
    x = centro[0] + radio*np.cos(u)*np.sin(v)
    y = centro[1] + radio*np.sin(u)*np.sin(v)
    z = centro[2] + radio*np.cos(v)
    ax.plot_surface(x, y, z, color=color, shade=True, linewidth=0)


def puntos_robot(q):
    frames = robot.fkine_all(q)          
    puntos = [T.t for T in frames]
    puntos.append(robot.fkine(q).t)      
    return puntos


def dibujar_robot(ax, q):
    puntos = puntos_robot(q)
    colores = ['#888888', '#3b6ea5', '#5aa469', '#c97a3d', '#a05ac9', '#333333']
    radios  = [0.018,     0.015,     0.013,     0.009,     0.007,     0.006]
    for i in range(len(puntos) - 1):
        cilindro(ax, puntos[i], puntos[i + 1], radios[i], colores[i])
    for p in puntos:
        esfera(ax, p, 0.02, '#222222')


plt.rcParams['figure.figsize'] = (8, 8)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
dibujar_robot(ax, q0)
ax.set_xlim(-0.05, 0.35); ax.set_ylim(-0.2, 0.2); ax.set_zlim(0, 0.4)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Brazo 6 GDL — reposo')
ax.view_init(elev=0, azim=-90)  # vista lateral
plt.show()

## Otra configuración 


In [88]:
q1 = [np.radians(19.8), np.radians(59.3), np.radians(-34.6), np.radians(-54.4), 0] 

T1 = robot.fkine(q1)
print(T1)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
dibujar_robot(ax, q1)
ax.set_xlim(-0.05, 0.3); ax.set_ylim(-0.15, 0.2); ax.set_zlim(0, 0.35)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Brazo 6 GDL ')
ax.view_init(elev=0, azim=-90)  # vista lateral
plt.show()

   0.8173    0.3387   -0.4662    0.2097    
   0.2942   -0.9409   -0.1678    0.07548   
  -0.4955    0        -0.8686    0.2284    
   0         0         0         1         



## movimiento 

In [89]:
from roboticstoolbox import jtraj

traj = jtraj(q0, q1, 10)

fig = plt.figure(figsize=(20, 5))
for i, q in enumerate(traj.q):
    ax = fig.add_subplot(2, 5, i + 1, projection='3d')
    dibujar_robot(ax, q)
    ax.set_xlim(-0.05, 0.3); ax.set_ylim(-0.15, 0.2); ax.set_zlim(0, 0.35)
    ax.view_init(elev=0, azim=-90)  # vista lateral
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.set_title(f'{i + 1}', fontsize=10)

plt.tight_layout()
plt.show()